# Notebook 04 — Statistical Analysis

**Objective:** Apply hypothesis tests, correlation analysis, and logistic regression to identify statistically significant predictors of startup closure. Each test is followed by a short interpretation in business language.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, pointbiserialr, norm
import statsmodels.formula.api as smf

from pathlib import Path
import os
import warnings

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(
    os.environ.get(
        'PROJECT_ROOT',
        Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
    )
).resolve()

def first_existing(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

RAW_DATA_PATH = first_existing(
    PROJECT_ROOT / 'data' / 'raw' / 'investments_VC.csv',
    PROJECT_ROOT / 'investments_VC.csv',
    Path('/content/investments_VC.csv'),
)

CLEAN_DATA_PATH = first_existing(
    PROJECT_ROOT / 'data' / 'processed' / 'startups_cleaned.csv',
    PROJECT_ROOT / 'startups_cleaned.csv',
    Path('/content/startups_cleaned.csv'),
)

SCREENSHOT_DIR = PROJECT_ROOT / 'tableau' / 'screenshots'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    _colab_files = None
    IN_COLAB = False

def maybe_download(path: Path) -> None:
    if IN_COLAB:
        _colab_files.download(str(path))

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

df = pd.read_csv(CLEAN_DATA_PATH, low_memory=False)
df = df.rename(columns={c: c.lower() if c.startswith('round_') else c for c in df.columns})

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('Statistical analysis initialized.')

def save_chart(fig, filename: str) -> Path:
    output_path = SCREENSHOT_DIR / filename
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    maybe_download(output_path)
    return output_path


In [ ]:
contingency = pd.crosstab(df['status'], df['reached_series_a'])
print('Contingency Table — Status vs. Reached Series A:')
print(contingency.to_string())

chi2, p_value, dof, expected = chi2_contingency(contingency)
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print(f'\nChi-Square Statistic : {chi2:.4f}')
print(f'Degrees of Freedom   : {dof}')
print(f'P-value              : {p_value:.6g}')
print(f"Cramér's V           : {cramers_v:.4f}")

### Finding — Chi-Square: Status vs. Series A
**Test:** Chi-square test of independence.
**Result:** `χ² = 935.42`, `p < 0.001`, `Cramér's V = 0.167` -> reject H₀.
**Business interpretation:** Reaching Series A is not independent of startup outcome. The association is statistically significant but only moderate in size, so Series A should be treated as an important signal within a broader risk framework rather than a one-variable verdict.

In [ ]:
closed_funding = df[(df['status'] == 'closed') & (df['funding_total_usd'] > 0)]['funding_total_usd'].dropna()
operating_funding = df[(df['status'] == 'operating') & (df['funding_total_usd'] > 0)]['funding_total_usd'].dropna()

print(f'Closed startups    — n={len(closed_funding):,}, median=${np.median(closed_funding):,.0f}')
print(f'Operating startups — n={len(operating_funding):,}, median=${np.median(operating_funding):,.0f}')

u_stat, p_val = mannwhitneyu(closed_funding, operating_funding, alternative='less')
z_score = norm.ppf(p_val)
r_effect = abs(z_score) / np.sqrt(len(closed_funding) + len(operating_funding))

print(f'\nMann-Whitney U      : {u_stat:.2f}')
print(f'P-value (one-sided): {p_val:.6g}')
print(f'Effect size r      : {r_effect:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
plot_data = [np.log10(closed_funding + 1), np.log10(operating_funding + 1)]
bp = ax.boxplot(plot_data, patch_artist=True, labels=['Closed', 'Operating'], medianprops={'color': 'black', 'linewidth': 2})
bp['boxes'][0].set_facecolor('#E74C3C')
bp['boxes'][1].set_facecolor('#2ECC71')
for patch in bp['boxes']:
    patch.set_alpha(0.7)
ax.set_title('Funding Distribution — Closed vs Operating (log₁₀ scale)', fontweight='bold', fontsize=13)
ax.set_ylabel('log₁₀(Total Funding USD)')
ax.text(0.98, 0.02, 'p < 0.001', transform=ax.transAxes, ha='right', fontsize=11, color='navy')
plt.tight_layout()
save_chart(fig, 'stat_funding_boxplot.png')
plt.show()

### Finding — Mann-Whitney U: Funding vs. Status
**Test:** Mann-Whitney U (non-parametric; funding is heavily right-skewed).
**Result:** `U = 14,728,907`, `p < 0.001`, effect size `r = 0.066` -> reject H₀. Median funding for closed startups is about **$850K** versus **$1.75M** for operating startups.
**Business interpretation:** Closed startups raise significantly less capital than operating ones. The effect size is modest, but the direction is consistent and business-relevant: underfunding is a measurable failure signal.

In [ ]:
status_order = ['closed', 'operating', 'acquired', 'ipo']
nonempty_groups = []
for status in status_order:
    group = df[df['status'] == status]['funding_rounds'].dropna()
    if len(group) > 0:
        nonempty_groups.append((status, group))

print('Median funding rounds by status:')
for name, grp in nonempty_groups:
    print(f'  {name:<12}: n={len(grp):,}, median={grp.median():.1f}, mean={grp.mean():.2f}')

h_stat, p_val_kw = kruskal(*[grp for _, grp in nonempty_groups])
print(f'\nKruskal-Wallis H   : {h_stat:.4f}')
print(f'P-value            : {p_val_kw:.6g}')

### Finding — Kruskal-Wallis: Funding Rounds by Outcome
**Test:** Kruskal-Wallis across the non-empty status groups (`closed`, `operating`, `acquired`).
**Result:** `H = 413.30`, `p < 0.001` -> reject H₀. Acquired startups have the strongest funding-round profile (mean about **2.10** rounds), while closed startups average about **1.45** rounds.
**Business interpretation:** Sustained investor follow-on is statistically linked to better outcomes. The number of rounds looks like a stronger survival signal than a one-time raise.

In [ ]:
reg_cols = ['is_closed', 'funding_rounds', 'funding_total_usd', 'reached_series_a', 'days_to_first_funding', 'is_usa']
df_reg = df[reg_cols].dropna()
df_reg = df_reg[df_reg['funding_total_usd'] > 0].copy()
df_reg['log_funding'] = np.log10(df_reg['funding_total_usd'])

print(f'Regression sample size: {len(df_reg):,}')
logit_model = smf.logit('is_closed ~ funding_rounds + log_funding + reached_series_a + is_usa', data=df_reg).fit(disp=0)
print(logit_model.summary2())

params = logit_model.params
conf = logit_model.conf_int()
p_vals = logit_model.pvalues
odds_df = pd.DataFrame({
    'Odds Ratio': np.exp(params),
    '95% CI Lower': np.exp(conf[0]),
    '95% CI Upper': np.exp(conf[1]),
    'P-value': p_vals,
    'Significant': p_vals < 0.05,
}).drop('Intercept')

print('\nLogistic Regression — Odds Ratios for Startup Closure:')
print(odds_df.round(4).to_string())

### Finding — Logistic Regression: Independent Predictors of Closure
**Test:** Logistic regression on `28,213` startups with complete regression features.
**Result:** `funding_rounds` (`OR = 0.786`, `p < 0.001`) and `log_funding` (`OR = 0.701`, `p < 0.001`) are protective. `is_usa` is weakly positive (`OR = 1.124`, `p = 0.040`), and `reached_series_a` is positive after controls (`OR = 1.679`, `p < 0.001`).
**Business interpretation:** The model confirms that more rounds and more capital reduce closure odds independently. The positive controlled coefficient for `reached_series_a` should be interpreted cautiously because it overlaps with funding-stage and scale variables; in other words, this feature is informative but not cleanly separable from the rest of the capital path.

In [ ]:
df_corr = df[['is_closed', 'funding_total_usd', 'funding_rounds', 'reached_series_a', 'avg_funding_per_round']].dropna()
df_corr = df_corr[df_corr['funding_total_usd'] > 0]

print('Point-Biserial Correlations with is_closed:')
print('=' * 55)
for col in ['funding_total_usd', 'funding_rounds', 'reached_series_a', 'avg_funding_per_round']:
    r_val, p_val = pointbiserialr(df_corr['is_closed'], df_corr[col])
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f'  {col:<30}: r = {r_val:+.4f}, p = {p_val:.4g} {sig}')

### Finding — Point-Biserial Correlations with Closure
**Test:** Point-biserial correlation between `is_closed` and key numeric predictors.
**Result:** `funding_total_usd` (`r = -0.0436`, `p < 0.001`), `funding_rounds` (`r = -0.0556`, `p < 0.001`), and `avg_funding_per_round` (`r = -0.0343`, `p < 0.001`) are all negatively associated with closure. `reached_series_a` is weak and not conventionally significant in this bivariate view (`p ≈ 0.066`).
**Business interpretation:** Capital variables move in the expected direction, but the effect sizes are small, which reinforces the idea that startup failure is multi-causal. No single number explains closure on its own.

In [ ]:
top_sectors = df['market'].value_counts().head(10).index.tolist()
df_sector = df[df['market'].isin(top_sectors)]
sector_groups = [df_sector[df_sector['market'] == sector]['is_closed'] for sector in top_sectors]

h_stat_sector, p_val_sector = kruskal(*sector_groups)
print('Kruskal-Wallis across top 10 sectors:')
print(f'  H = {h_stat_sector:.4f}, p = {p_val_sector:.6g}')

sector_summary = df_sector.groupby('market').agg(count=('is_closed', 'count'), failure_rate=('is_closed', 'mean')).sort_values('failure_rate', ascending=False)
sector_summary['failure_rate_pct'] = (sector_summary['failure_rate'] * 100).round(1)
print('\nFailure rates by top sector:')
print(sector_summary[['count', 'failure_rate_pct']].to_string())

### Finding — Kruskal-Wallis: Sector-Level Failure Differences
**Test:** Kruskal-Wallis across the ten largest markets in the dataset.
**Result:** `H = 232.82`, `p < 0.001` -> reject H₀. Within the top sectors, `Curated Web` and `Games` sit near the high-risk end, while `Biotechnology` and `Enterprise Software` are materially lower.
**Business interpretation:** Sector selection changes the base rate of failure. Investors should not evaluate every startup against one universal hurdle rate; the market context needs to be priced into the decision.

In [ ]:
total = len(df)
closed = (df['status'] == 'closed').sum()
acquired = (df['status'] == 'acquired').sum()
ipo_count = (df['status'] == 'ipo').sum()
operating = (df['status'] == 'operating').sum()

kpis = {
    'Overall Failure Rate (%)': round(closed / total * 100, 2),
    'Acquisition Success Rate (%)': round(acquired / total * 100, 2),
    'IPO Rate (%)': round(ipo_count / total * 100, 2),
    'Operating Rate (%)': round(operating / total * 100, 2),
    'Median Funding — Closed ($)': int(df[df['status'] == 'closed']['funding_total_usd'].median()),
    'Median Funding — Operating ($)': int(df[df['status'] == 'operating']['funding_total_usd'].median()),
    'Avg Funding Rounds — Closed': round(df[df['status'] == 'closed']['funding_rounds'].mean(), 2),
    'Avg Funding Rounds — Operating': round(df[df['status'] == 'operating']['funding_rounds'].mean(), 2),
    'Series A Failure Rate (%)': round(df[df['reached_series_a'] == 1]['is_closed'].mean() * 100, 2),
    'Non-Series A Failure Rate (%)': round(df[df['reached_series_a'] == 0]['is_closed'].mean() * 100, 2),
}

print('=== PROJECT KPI DASHBOARD ===')
for k, v in kpis.items():
    print(f'  {k:<40}: {v}')